<a href="https://colab.research.google.com/github/Thilac01/Statistical-Learning-e22395/blob/main/Statistical_Learning_Assignement4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Question 1: The Total Variance Illusion (PCA vs. FA Subspace Allocations)

1. **Physical Nature of Sensor 4:** A Uniqueness value ($\varphi^2$) exceeding $98\%$ means that less than $2\%$ of the variance in `Sensor_4` is driven by the shared common factors ($f_1, f_2$). This proves that `Sensor_4` is completely decoupled from the shared structural dynamics of the asset. Physically, this confirms that the variations recorded by `Sensor_4` represent purely localized white noise, such as a major hardware calibration error or an electrical instrumentation short, rather than actual structural behavior.
2. **The PCA Illusion:** PCA operates by maximizing global variance without distinguishing between shared structural variation and localized sensor noise. The variance of `Sensor_4` is exceptionally high ($\sigma^2 \approx 2.0$) due to its noise level. To capture this large total variance, the PCA engine rotates its top principal axes toward `Sensor_4`, which artificially inflates the top eigenvalues ($\lambda_1, \lambda_2$) and absorbs this localized noise directly into the primary "clean" subspace.
3. **Operational Monitoring Risks:** If an engineer relies solely on a PCA monitoring pipeline, this creates a major vulnerability. Because the principal subspace is heavily contaminated by the noise of a single malfunctioning sensor (`Sensor_4`), the Hotelling's $T^2$ boundary will regularly trigger false alarms from harmless electrical spikes. Conversely, true structural failures (which are captured by `Sensor_1` and `Sensor_2`) can be masked or ignored because their variance contribution is small relative to the noise of `Sensor_4`.

---

# **Subspace Diagnostics & Feature De-correlation: Case Analysis**

This notebook evaluates two fundamental methodologies for multi-sensor asset diagnostics:
1. **Principal Component Analysis (PCA):** Maximizes global variance non-parametrically.
2. **Factor Analysis (FA):** Parametrically partitions shared structural modes from localized instrument noise.

---

## **Question 1: The Total Variance Illusion (PCA vs. FA Subspace Allocations)**



* **Physical Nature of Sensor 4:** A Uniqueness value ($\varphi^2 > 98\%$) proves that less than $2\%$ of its variance is driven by the structural processes ($f_1, f_2$). It is completely decoupled from the system and represents purely localized white noise (e.g., an electrical short or hardware calibration fault).
* **The PCA Illusion:** PCA maximizes total global variance without distinguishing its source. Because `Sensor_4` has a massive noise variance ($\sigma^2 \approx 2.0$), the PCA engine rotates its top principal components directly toward this noise axis, artificially inflating its top eigenvalues and contaminating the "clean" subspace.
* **Operational Monitoring Risks:** Relying solely on a raw PCA pipeline creates high false-alarm rates because harmless electrical spikes instantly trip the Hotelling's $T^2$ boundary. Simultaneously, true structural degradation (tracked by `Sensor_1` and `Sensor_2`) gets masked or ignored because its energy is small compared to the noise floor.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA, FactorAnalysis

# 1. Regenerate assignment data profile
np.random.seed(42)
n_samples = 2500
f1 = np.random.normal(0, 1, n_samples)
f2 = np.random.normal(0, 1, n_samples)

s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples) # High noise floor

df_asset = pd.DataFrame(np.vstack([s1, s2, s3, s4]).T, columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4'])

# Standardize observations
Z = (df_asset - df_asset.mean()) / df_asset.std()

# 2. Compare Variance Attributions
pca = PCA().fit(Z)
fa = FactorAnalysis(n_components=2).fit(Z)

print("--- QUESTION 1 MATHEMATICAL VERIFICATION ---")
print(f"PCA PC1 + PC2 Explained Variance Ratio: {np.sum(pca.explained_variance_ratio_[:2])*100:.2f}%")
print(f"FA Isolated Sensor_4 Uniqueness (Noise Floor): {fa.noise_variance_[3]*100:.2f}%")
print("\nConclusion: PCA captures the noise of Sensor 4 as structural information, while FA isolates it.")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import PCA, FactorAnalysis

# --- 1. Data Setup & Core Models ---
np.random.seed(42)
n_samples = 2500
f1, f2 = np.random.normal(0, 1, n_samples), np.random.normal(0, 1, n_samples)
s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)
df_asset = pd.DataFrame(np.vstack([s1, s2, s3, s4]).T, columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4'])
Z = (df_asset - df_asset.mean()) / df_asset.std()

pca = PCA().fit(Z)
fa = FactorAnalysis(n_components=2).fit(Z)

# --- 2. Inline Varimax Rotation Engine ---
def varimax(L):
    p, k = L.shape
    R = np.eye(k)
    d = 0
    for _ in range(500):
        old_d = d
        L_r = L @ R
        alpha = np.diag(np.sum(L_r**2, axis=0))
        B = L.T @ (L_r**3 - (1.0 / p) * L_r @ alpha)
        U, S, Vt = np.linalg.svd(B)
        R = U @ Vt
        d = np.sum(S)
        if old_d != 0 and (d - old_d) / old_d < 1e-6: break
    return L @ R

raw_pca_loadings = pca.components_.T[:, :2]
rotated_fa_loadings = varimax(fa.components_.T)

print("[Raw Unrotated PCA Weights]")
print(pd.DataFrame(raw_pca_loadings, index=df_asset.columns, columns=['PC1', 'PC2']).round(3))
print("\n[Varimax-Rotated FA Weights (Simple Structure)]")
print(pd.DataFrame(rotated_fa_loadings, index=df_asset.columns, columns=['Factor 1', 'Factor 2']).round(3))

# --- 3. Render Absolute Structural Loadings Heatmap ---
fig = go.Figure(data=go.Heatmap(
    z=np.abs(rotated_fa_loadings),
    x=["Factor 1", "Factor 2"],
    y=list(df_asset.columns),
    colorscale='YlOrRd',
    colorbar=dict(title="Absolute Weight")
))
fig.update_layout(title="<b>Structural Loadings Matrix Heatmap (|λ|)</b>", template='plotly_white', width=600, height=400)
fig.show()

---

## **Question 2: Decoupling Structural Loading via Rotation (FA Subplot 1 vs. PCA Eigenvectors)**



* **Mathematical Mechanics of Varimax:** Traditional PCA forces eigenvectors to follow a rigid mathematical hierarchy ($\lambda_1 > \lambda_2 > \dots$), often spreading mixed, uninterpretable weights across all sensors on the first component. Factor Analysis relaxes this hierarchy and uses an orthogonal **Varimax rotation** to maximize the variance of the squared loadings down each column, pulling weights cleanly toward $0$ or $\pm 1$.
* **Operator Troubleshooting Advantage:** Varimax creates a **Simple Structure** that groups distinct sensors onto independent factors. Instead of a mixed PCA vector, the operator gets an explicit breakdown: `Sensor_1` and `Sensor_2` load onto `Factor 1` (Primary Structural Mode), while `Sensor_3` maps to `Factor 2` (Operational Mode). If `Factor 1` spikes, the operator knows instantly which specific physical components to inspect.

In [ ]:
# Varimax Rotation Utility
def apply_varimax(loadings, max_iter=500, tol=1e-6):
    p, k = loadings.shape
    R = np.eye(k)
    d = 0
    for i in range(max_iter):
        old_d = d
        L = loadings @ R
        alpha = np.diag(np.sum(L**2, axis=0))
        B = loadings.T @ (L**3 - (1.0 / p) * L @ alpha)
        U, S, Vt = np.linalg.svd(B)
        R = U @ Vt
        d = np.sum(S)
        if old_d != 0 and (d - old_d) / old_d < tol: break
    return loadings @ R

print("--- QUESTION 2 MATRIX RECONSTRUCTION ---")
raw_pca_loadings = pca.components_.T[:, :2]
rotated_fa_loadings = apply_varimax(fa.components_.T)

print("\n[Raw Unrotated PCA Subspace Weights (PC1 vs PC2)]")
print(pd.DataFrame(raw_pca_loadings, index=df_asset.columns, columns=['PC1', 'PC2']).round(3))

print("\n[Varimax-Rotated FA Subspace Weights (Factor 1 vs Factor 2)]")
print(pd.DataFrame(rotated_fa_loadings, index=df_asset.columns, columns=['Factor 1', 'Factor 2']).round(3))

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import PCA, FactorAnalysis

# --- 1. Data Setup & Core Models ---
np.random.seed(42)
n_samples = 2500
f1, f2 = np.random.normal(0, 1, n_samples), np.random.normal(0, 1, n_samples)
s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)
df_asset = pd.DataFrame(np.vstack([s1, s2, s3, s4]).T, columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4'])
Z = (df_asset - df_asset.mean()) / df_asset.std()

pca = PCA().fit(Z)
fa = FactorAnalysis(n_components=2).fit(Z)

# --- 2. Inline Varimax Rotation Engine ---
def varimax(L):
    p, k = L.shape
    R = np.eye(k)
    d = 0
    for _ in range(500):
        old_d = d
        L_r = L @ R
        alpha = np.diag(np.sum(L_r**2, axis=0))
        B = L.T @ (L_r**3 - (1.0 / p) * L_r @ alpha)
        U, S, Vt = np.linalg.svd(B)
        R = U @ Vt
        d = np.sum(S)
        if old_d != 0 and (d - old_d) / old_d < 1e-6: break
    return L @ R

raw_pca_loadings = pca.components_.T[:, :2]
rotated_fa_loadings = varimax(fa.components_.T)

print("[Raw Unrotated PCA Weights]")
print(pd.DataFrame(raw_pca_loadings, index=df_asset.columns, columns=['PC1', 'PC2']).round(3))
print("\n[Varimax-Rotated FA Weights (Simple Structure)]")
print(pd.DataFrame(rotated_fa_loadings, index=df_asset.columns, columns=['Factor 1', 'Factor 2']).round(3))

# --- 3. Render Absolute Structural Loadings Heatmap ---
fig = go.Figure(data=go.Heatmap(
    z=np.abs(rotated_fa_loadings),
    x=["Factor 1", "Factor 2"],
    y=list(df_asset.columns),
    colorscale='YlOrRd',
    colorbar=dict(title="Absolute Weight")
))
fig.update_layout(title="<b>Structural Loadings Matrix Heatmap (|λ|)</b>", template='plotly_white', width=600, height=400)
fig.show()

---

## **Question 3: Determining Subspace Truncation ($k$) using $T^2$ and $Q$ Profiles**



* **Curve Profile Trajectory:** Moving from $k=1$ to $k=2$ causes a sharp, massive drop in the Mean Residual $Q$ statistic, confirming that the second component accounts for genuine physical energy. Transitioning from $k=2$ to $k=3$ results in a completely flat, horizontal "elbow."
* **Identifying True Dimensionality ($k=2$):** The distinct elbow at $k=2$ separates true structured physical process variations from random instrument noise. It confirms that both structural underlying dimensions ($f_1, f_2$) are fully captured within the active subspace.
* **Consequences of Over-fitting ($k=3$):** If an engineer scales the monitoring subspace to $k=3$, **pure localized sensor noise is forced into the "clean" subspace**. This degrades Hotelling's $T^2$ by making it track random instrument drift, while leaving the residual $Q$ monitor under-sensitized and blind to real structural failures.

In [ ]:
print("--- QUESTION 3 METRIC TRUNCATION HIGHLIGHTS ---")
Z_proj = Z.to_numpy() @ pca.components_.T
summary_stack = []

for k in range(1, 4):
    t2 = np.mean(np.sum((Z_proj[:, :k]**2) / pca.explained_variance_[:k], axis=1))
    q = np.mean(np.sum(Z_proj[:, k:]**2, axis=1))
    summary_stack.append({"Cutoff (k)": k, "Mean Hotelling T2": round(t2, 3), "Mean Residual Q": round(q, 3)})

print(pd.DataFrame(summary_stack).to_string(index=False))
print("\nObservation: The residual error (Q) drops drastically at k=2 and plateaus, identifying k=2 as the optimal cutoff.")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import PCA

# --- 1. Data Setup & Principal Projections ---
np.random.seed(42)
n_samples = 2500
f1, f2 = np.random.normal(0, 1, n_samples), np.random.normal(0, 1, n_samples)
s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)
df_asset = pd.DataFrame(np.vstack([s1, s2, s3, s4]).T, columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4'])
Z = (df_asset - df_asset.mean()) / df_asset.std()

pca = PCA().fit(Z)
lambdas = pca.explained_variance_
Z_proj = Z.to_numpy() @ pca.components_.T

k_values = [1, 2, 3, 4]
t2_profile, q_profile = [], []

# --- 2. Calculate Diagnostic Energies across Subspaces ---
for k in k_values:
    t2_val = np.mean(np.sum((Z_proj[:, :k]**2) / lambdas[:k], axis=1))
    q_val = np.mean(np.sum(Z_proj[:, k:]**2, axis=1)) if k < 4 else 0
    t2_profile.append(t2_val)
    q_profile.append(q_val)

# --- 3. Plot Residual Q Decay and Subspace Elbow Curve ---
fig = go.Figure()
fig.add_trace(go.Scatter(x=k_values, y=q_profile, mode='lines+markers', name='Mean Q (SPE)', line=dict(color='#d62728', width=2)))
fig.add_trace(go.Scatter(x=k_values, y=t2_profile, mode='lines+markers', name="Mean Hotelling's T²", line=dict(color='#1f77b4', dash='dash')))

fig.update_layout(
    title="<b>Subspace Diagnostic Profiles vs. Subspace Size (k)</b>",
    xaxis=dict(title="Subspace Truncation Cutoff (k)", tickmode='linear', tick0=1, dtick=1),
    yaxis=dict(title="Metric Statistical Energy Mean"),
    template="plotly_white", width=800, height=400
)
fig.show()

---

## **Question 4: Operational Trade-offs in System Health Monitoring**



* **Strategy Comparison:**
  * **PCA Strategy ($T^2 + Q$):** Comprehensive because it monitors both in-subspace scale ($T^2$) and out-of-subspace model violations ($Q$), but highly vulnerable to single-point sensor failures.
  * **FA Strategy (Factor Scores):** Purely isolates and monitors the actual underlying physical driving forces ($f_1, f_2$) while explicitly dropping the sensor noise floor.
* **Robustness Choice:** The **Factor Analysis Strategy** is significantly more robust against sensor calibration loss or electrical failures.

#### **Technical Justification**
FA defines the system matrix as $\mathbf{R} = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$, where $\boldsymbol{\Psi}$ isolates unique sensor noise on the diagonal. Thomson’s score projection maps the data via:

$$\mathbf{F} = \mathbf{Z} \mathbf{R}^{-1} \boldsymbol{\Lambda}_{\text{rotated}}$$

When a sensor breaks down and its variance spikes, its uniqueness parameter ($\varphi^2$) approaches $1.0$. This causes its corresponding entry in $\mathbf{R}^{-1}$ to collapse toward zero. The FA engine automatically down-weights and excludes the broken tracking channel, preventing its noise from bleeding into the factor scores. The plant operator can continue safely tracking the asset's true physical status using the remaining operational sensors.

In [ ]:
print("--- QUESTION 4 MATHEMATICAL VERIFICATION (THOMSON SCORES) ---")
# Extract Inverse Correlation Matrix and verify weight distribution
R_inv = np.linalg.inv((Z.T @ Z) / (n_samples - 1))
thomson_weights = R_inv @ rotated_fa_loadings

print("[Thomson Score Linear Projection Weights Matrix]")
print(pd.DataFrame(thomson_weights, index=df_asset.columns, columns=['Weight F1', 'Weight F2']).round(3))
print("\nResult: Notice how Sensor_4's projection weights collapse close to zero automatically, isolating the channel noise!")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import FactorAnalysis

# --- 1. Data Setup & Factor Extraction ---
np.random.seed(42)
n_samples = 2500
f1, f2 = np.random.normal(0, 1, n_samples), np.random.normal(0, 1, n_samples)
s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)
df_asset = pd.DataFrame(np.vstack([s1, s2, s3, s4]).T, columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4'])
Z = (df_asset - df_asset.mean()) / df_asset.std()

fa = FactorAnalysis(n_components=2).fit(Z)

# --- 2. Inline Varimax ---
def varimax(L):
    p, k = L.shape; R = np.eye(k); d = 0
    for _ in range(500):
        old_d = d; L_r = L @ R; alpha = np.diag(np.sum(L_r**2, axis=0))
        B = L.T @ (L_r**3 - (1.0 / p) * L_r @ alpha); U, S, Vt = np.linalg.svd(B); R = U @ Vt; d = np.sum(S)
        if old_d != 0 and (d - old_d) / old_d < 1e-6: break
    return L @ R

rotated_loadings = varimax(fa.components_.T)
uniqueness = fa.noise_variance_

# --- 3. Construct Thomson Score Projection Matrix ---
R_matrix = (Z.T @ Z) / (n_samples - 1)
R_inv = np.linalg.inv(R_matrix)
thomson_weights = R_inv @ rotated_loadings

print("--- THOMSON SCORE PROJECTION WEIGHTS ---")
df_weights = pd.DataFrame(thomson_weights, index=df_asset.columns, columns=['Weight_Factor_1', 'Weight_Factor_2'])
print(df_weights.round(3))

# --- 4. Plot Sensor Uniqueness Noise Floor Profile ---
fig = go.Figure(data=go.Scatter(
    x=list(df_asset.columns), y=uniqueness,
    mode='lines+markers', name='Uniqueness Profile (φ²)',
    line=dict(color='#d62728', width=2, dash='dashdot'),
    marker=dict(symbol='x', size=10)
))
fig.update_layout(
    title="<b>Sensor Uniqueness Noise Floor Profile (Instrument Fault Isolation)</b>",
    xaxis=dict(title="Monitored Sensor Channels", tickangle=25),
    yaxis=dict(title="Uniqueness Value (φ²)", range=[0, 1.05]),
    template='plotly_white', width=800, height=400
)
fig.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.subplots as sp
import plotly.graph_objects as go
from sklearn.decomposition import PCA, FactorAnalysis

# --- REGENERATE DATA ---
np.random.seed(42)
n_samples = 2500
f1 = np.random.normal(0, 1, n_samples)
f2 = np.random.normal(0, 1, n_samples)

s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)

df_asset = pd.DataFrame(np.vstack([s1, s2, s3, s4]).T, columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4'])
Z = (df_asset - df_asset.mean()) / df_asset.std()

# --- COMPUTE PCA METRICS ---
pca = PCA().fit(Z)
lambdas = pca.explained_variance_
var_exp = pca.explained_variance_ratio_ * 100
cum_var = np.cumsum(var_exp)
Z_proj = Z.to_numpy() @ pca.components_.T

t2_dim, q_dim = [], []
for k in range(1, 5):
    t2_dim.append(np.mean(np.sum((Z_proj[:, :k]**2) / lambdas[:k], axis=1)))
    q_dim.append(np.mean(np.sum(Z_proj[:, k:]**2, axis=1)) if k < 4 else 0)

# --- COMPUTE FA METRICS ---
fa = FactorAnalysis(n_components=2).fit(Z)
# Inline Varimax
def varimax(L):
    p, k = L.shape; R = np.eye(k); d = 0
    for _ in range(500):
        old_d = d; L_r = L @ R; alpha = np.diag(np.sum(L_r**2, axis=0))
        B = L.T @ (L_r**3 - (1.0 / p) * L_r @ alpha); U, S, Vt = np.linalg.svd(B); R = U @ Vt; d = np.sum(S)
        if old_d != 0 and (d - old_d) / old_d < 1e-6: break
    return L @ R

rotated_loadings = varimax(fa.components_.T)
communalities = np.sum(rotated_loadings**2, axis=1)
uniqueness = fa.noise_variance_
F_scores = Z.to_numpy() @ (np.linalg.inv((Z.T @ Z) / (n_samples - 1)) @ rotated_loadings)
fa_variances = np.var(F_scores, axis=0, ddof=1)

# ==========================================
# RENDER CANVAS 1: PCA OPTIMIZATION SUITE
# ==========================================
fig1 = sp.make_subplots(rows=2, cols=3, subplot_titles=(
    "Feature Loadings Matrix |P|", "Absolute Eigenvalues (λ)", "Explained Variance Ratio",
    "Residual Unexplained Space", "Mean Hotelling's T² vs. k", "Mean Q Statistic vs. k"
))
fig1.add_trace(go.Heatmap(z=np.abs(pca.components_), colorscale='Viridis', showscale=False), row=1, col=1)
fig1.add_trace(go.Bar(x=[f"PC{i}" for i in range(1,5)], y=lambdas, marker_color='#1f77b4'), row=1, col=2)
fig1.add_trace(go.Bar(x=[f"PC{i}" for i in range(1,5)], y=var_exp, name="Marginal"), row=1, col=3)
fig1.add_trace(go.Scatter(x=[f"PC{i}" for i in range(1,5)], y=cum_var, name="Cumulative", line=dict(dash='dash', color='black')), row=1, col=3)
fig1.add_trace(go.Bar(x=[f"PC{i}" for i in range(1,5)], y=100 - cum_var, marker_color='#d62728'), row=2, col=1)
fig1.add_trace(go.Scatter(x=[1,2,3,4], y=t2_dim, mode='lines+markers', name="T²"), row=2, col=2)
fig1.add_trace(go.Scatter(x=[1,2,3,4], y=q_dim, mode='lines+markers', name="Q"), row=2, col=3)
fig1.update_layout(title="<b>INTERFACE LAYOUT 1: PCA OPTIMIZATION SUITE</b>", template="plotly_white", width=1250, height=750)
fig1.show()

# ==========================================
# RENDER CANVAS 2: FA LATENT SUBSPACE SUITE
# ==========================================
fig2 = sp.make_subplots(rows=2, cols=2, horizontal_spacing=0.24, vertical_spacing=0.28, subplot_titles=(
    "Structural Loadings Matrix Heatmap", "Variance Partitioning Profile",
    "Sensor Uniqueness Line Profile", "Latent Empirical Variance"
))
fig2.add_trace(go.Heatmap(z=np.abs(rotated_loadings), x=["Factor 1", "Factor 2"], y=list(df_asset.columns), colorscale='YlOrRd', colorbar=dict(x=-0.15, len=0.38, y=0.78, yanchor="middle", xanchor="right")), row=1, col=1)
fig2.add_trace(go.Bar(y=list(df_asset.columns), x=communalities * 100, name="Shared Communality (h²)", orientation='h', marker_color='#1f77b4'), row=1, col=2)
fig2.add_trace(go.Bar(y=list(df_asset.columns), x=uniqueness * 100, name="Isolated Uniqueness (φ²)", orientation='h', marker_color='#ff7f0e'), row=1, col=2)
fig2.add_trace(go.Scatter(x=list(df_asset.columns), y=uniqueness, mode='lines+markers', name='Uniqueness (φ²)', line=dict(color='#d62728', width=2, dash='dashdot'), marker=dict(symbol='x', size=8)), row=2, col=1)
fig2.add_trace(go.Bar(x=["Factor 1", "Factor 2"], y=fa_variances, name='Factor Variance', marker=dict(color='#2ca02c', line=dict(color='black', width=0.5))), row=2, col=2)
fig2.update_layout(title="<b>INTERFACE LAYOUT 2: FA LATENT SUBSPACE SUITE</b>", template='plotly_white', barmode='stack', width=1250, height=750, margin=dict(t=150, b=60, l=140, r=80), legend=dict(orientation="h", x=0.5, y=1.02, xanchor="center"))
fig2.update_xaxes(range=[0, 100], row=1, col=2)
fig2.update_xaxes(tickangle=25, row=2, col=1)
fig2.show()